# 2025 Coral Heat Stress Predictions

Loads the fold 5 XGBoost model (trained on 1985–2016, tested on 2017–2024) from
`stress_predictions.ipynb` and evaluates it on held-out 2025 data for all three
reef sites.  Fold 5 test-set performance (2017–2024) is also reprinted for reference.

**Author:** Marybeth Arcodia

**Usage:** Run all cells top-to-bottom.  Requires that `stress_predictions.ipynb`
has been run first so the fold-5 models exist under `MODELS_EXP` in `../output/`.

In [ ]:
import sys
import json
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from xgboost import XGBClassifier

import preprocess as preprocess_module
from data_split import create_labels_and_features
import train_test_model

warnings.filterwarnings('ignore')
np.set_printoptions(threshold=np.inf)

## Configuration

In [ ]:
# ── Sites ─────────────────────────────────────────────────────────────────────
SITES_TO_RUN = ['sombrero', 'molasses', 'sk']

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../Data')
OUTPUT_DIR = Path('../output')

# Source experiment: where the fold-5 models were saved by stress_predictions.ipynb.
# This must match the EXP_NAME used in that run.
MODELS_EXP = 'exp001'

# Output folder for 2025 prediction plots and results.
# Change this to whatever descriptive name you prefer.
OUT_EXP = 'predictions_2025'

MODEL_FOLD = 5

# ── Evaluation year and fold 5 reference years ────────────────────────────────
EVAL_YEAR   = 2025
FOLD5_YEARS = list(range(2017, 2025))   # 2017–2024

# ── Data file ─────────────────────────────────────────────────────────────────
DATA_FILENAME_TEMPLATE = '{site}_preds.csv'

# ── Prediction parameters (must match stress_predictions.ipynb) ───────────────
STRESS_THRESHOLD = 4
PREDICTION_SPAN  = 6
VAL_NO_ONSET     = PREDICTION_SPAN + 1   # = 7
SUMMER_START     = 21
SUMMER_END       = 40
WEEKS_PER_YEAR   = SUMMER_END - SUMMER_START + 1   # = 20
TOLERANCE        = 1

COLUMNS_TO_KEEP = [
    'week', 'meanDhw', 'meanHs', 'weeks_until_onset',
    'meanDhw_lag_1', 'meanDhw_lag_2', 'meanSsta',
    'airTempInter', 'zonalWindInter', 'oniInter',
    'loop_curr_sst', 'dswrf',
]

SITE_DISPLAY_NAMES = {'sombrero': 'Sombrero', 'sk': 'Sand Key', 'molasses': 'Molasses'}

print(f'Sites:             {SITES_TO_RUN}')
print(f'Models from:       {OUTPUT_DIR / MODELS_EXP}  (fold {MODEL_FOLD})')
print(f'Outputs to:        {OUTPUT_DIR / OUT_EXP}')
print(f'Evaluation year:   {EVAL_YEAR}')
print(f'Fold 5 ref years:  {FOLD5_YEARS[0]}–{FOLD5_YEARS[-1]}')

## Preprocessing

Runs the same pipeline as `stress_predictions.ipynb` but **without a row cutoff**,
so that 2025 data is included.  The `year` column is saved before `select_columns`
drops it, then used to split the processed data into fold-5 reference rows
(2017–2024) and the evaluation rows (2025).

In [ ]:
all_2025_results = {}

for reef_location in SITES_TO_RUN:
    site_name  = SITE_DISPLAY_NAMES.get(reef_location, reef_location.capitalize())
    model_path = (
        OUTPUT_DIR / MODELS_EXP / reef_location / 'models'
        / f'{reef_location}_xgb_fold_{MODEL_FOLD}.model'
    )

    print()
    print('=' * 60)
    print(f'Site: {site_name}')
    print(f'Model: {model_path}')

    # ── Preprocess all years (no row cutoff — we need 2025) ───────────────────
    preprocessor = preprocess_module.CoralBleachingPreprocessor(
        str(DATA_DIR) + '/',
        reef_location,
        STRESS_THRESHOLD,
        VAL_NO_ONSET,
        PREDICTION_SPAN,
        WEEKS_PER_YEAR,
    )
    preprocessor.load_data(DATA_FILENAME_TEMPLATE.format(site=reef_location))
    preprocessor.preprocess_time_variables()
    # No filter_data cutoff — include all years
    preprocessor.calculate_weeks_until_onset(PREDICTION_SPAN)
    preprocessor.add_lag_features({'meanDhw': [1, 2]})

    # Save year before select_columns drops it
    year_series = preprocessor.data['year'].copy()

    preprocessor.select_columns(COLUMNS_TO_KEEP)
    preprocessor.drop_missing_values()
    preprocessor.filter_summer_weeks(SUMMER_START, SUMMER_END)

    processed_data = preprocessor.data.copy()
    processed_data['year'] = year_series.loc[processed_data.index].values

    # ── Split into fold-5 reference (2017-2024) and eval (2025) ──────────────
    data_fold5 = processed_data[processed_data['year'].isin(FOLD5_YEARS)].drop(columns=['year'])
    data_2025  = processed_data[processed_data['year'] == EVAL_YEAR].drop(columns=['year'])

    print(f'Fold 5 rows (2017-2024): {len(data_fold5)}  |  2025 rows: {len(data_2025)}')

    if len(data_2025) == 0:
        print(f'WARNING: no {EVAL_YEAR} summer data found for {reef_location} — skipping.')
        continue

    # ── Load saved fold 5 model ───────────────────────────────────────────────
    if not model_path.exists():
        print(f'ERROR: model file not found: {model_path}')
        print(f'Run stress_predictions.ipynb with EXP_NAME="{MODELS_EXP}" first.')
        continue
    xgb_model = XGBClassifier()
    xgb_model.load_model(str(model_path))
    print('Model loaded.')

    # ── Predict on 2025 ──────────────────────────────────────────────────────
    labels_2025, features_2025 = create_labels_and_features(data_2025)
    preds_raw_2025 = np.round(xgb_model.predict(features_2025).astype(float), 0)
    masked_preds_2025 = train_test_model.apply_onset_mask_single(
        labels_2025, preds_raw_2025, WEEKS_PER_YEAR
    )

    # ── Metrics for 2025 ─────────────────────────────────────────────────────
    cutoff_2025 = np.repeat(VAL_NO_ONSET, len(labels_2025))
    valid_mask  = ~np.isnan(masked_preds_2025)

    hit  = valid_mask & (labels_2025 < VAL_NO_ONSET) & (np.abs(masked_preds_2025 - labels_2025) <= TOLERANCE)
    miss = valid_mask & (labels_2025 < VAL_NO_ONSET) & (np.abs(masked_preds_2025 - labels_2025) > TOLERANCE)
    fa   = valid_mask & (labels_2025 == VAL_NO_ONSET) & (masked_preds_2025 < cutoff_2025)
    tno  = valid_mask & (labels_2025 == VAL_NO_ONSET) & (masked_preds_2025 >= cutoff_2025)

    n_hit, n_miss = int(np.sum(hit)), int(np.sum(miss))
    n_fa,  n_tno  = int(np.sum(fa)),  int(np.sum(tno))

    precision = n_hit / (n_hit + n_fa)   if (n_hit + n_fa)   > 0 else float('nan')
    recall    = n_hit / (n_hit + n_miss) if (n_hit + n_miss) > 0 else float('nan')
    f1        = (2 * precision * recall / (precision + recall)
                 if (np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0)
                 else float('nan'))
    fa_rate   = n_fa / (n_fa + n_tno) if (n_fa + n_tno) > 0 else float('nan')
    onset_acc = n_hit / (n_hit + n_miss) if (n_hit + n_miss) > 0 else float('nan')

    onset_idx   = np.where(labels_2025 == 0)[0]
    onset_label = (f'onset at summer week {onset_idx[0] + 1} (calendar week {onset_idx[0] + SUMMER_START})'
                   if len(onset_idx) > 0 else 'no onset')
    print(f'  2025 truth:    {onset_label}')
    print(f'  Hit={n_hit}  Miss={n_miss}  FA={n_fa}  TNO={n_tno}')
    if np.isfinite(onset_acc):
        print(f'  Onset acc: {onset_acc:.0%}  FA rate: {fa_rate:.0%}  F1: {f1:.3f}')
    else:
        print(f'  No onset weeks in 2025.  FA rate: {fa_rate:.0%}')

    # ── Fold 5 verification (2017–2024) ──────────────────────────────────────
    labels_f5, features_f5 = create_labels_and_features(data_fold5)
    preds_f5   = np.round(xgb_model.predict(features_f5).astype(float), 0)
    masked_f5  = train_test_model.apply_onset_mask_single(labels_f5, preds_f5, WEEKS_PER_YEAR)
    valid_f5   = ~np.isnan(masked_f5)
    cutoff_f5  = np.repeat(VAL_NO_ONSET, len(labels_f5))

    hit_f5  = valid_f5 & (labels_f5 < VAL_NO_ONSET) & (np.abs(masked_f5 - labels_f5) <= TOLERANCE)
    miss_f5 = valid_f5 & (labels_f5 < VAL_NO_ONSET) & (np.abs(masked_f5 - labels_f5) > TOLERANCE)
    fa_f5   = valid_f5 & (labels_f5 == VAL_NO_ONSET) & (masked_f5 < cutoff_f5)
    tno_f5  = valid_f5 & (labels_f5 == VAL_NO_ONSET) & (masked_f5 >= cutoff_f5)

    n_h, n_m, n_f, n_t = (int(np.sum(hit_f5)), int(np.sum(miss_f5)),
                           int(np.sum(fa_f5)),  int(np.sum(tno_f5)))
    pr_f5 = n_h / (n_h + n_f) if (n_h + n_f) > 0 else 0
    re_f5 = n_h / (n_h + n_m) if (n_h + n_m) > 0 else 0
    f1_f5 = 2 * pr_f5 * re_f5 / (pr_f5 + re_f5) if (pr_f5 + re_f5) > 0 else 0
    print(f'  Fold5 ref (2017-24): Hit={n_h} Miss={n_m} FA={n_f} TNO={n_t}  F1={f1_f5:.3f}')

    all_2025_results[reef_location] = {
        'site_name':      site_name,
        'labels_2025':    labels_2025,
        'preds_raw_2025': preds_raw_2025,
        'masked_preds':   masked_preds_2025,
        'data_2025':      data_2025,
        'metrics_2025': {
            'hit': n_hit, 'miss': n_miss, 'fa': n_fa, 'tno': n_tno,
            'f1': f1, 'fa_rate': fa_rate, 'onset_acc': onset_acc,
        },
        'fold5_ref': {
            'labels': labels_f5,
            'preds':  preds_f5,
            'hit': n_h, 'miss': n_m, 'fa': n_f, 'tno': n_t, 'f1': f1_f5,
        },
    }

print()
print('Done. Run the cells below to view per-week output, plots, and the summary table.')

In [ ]:
if not all_2025_results:
    print('Run the prediction cell first.')
else:
    for reef_location in SITES_TO_RUN:
        if reef_location not in all_2025_results:
            continue

        res       = all_2025_results[reef_location]
        site_name = res['site_name']

        print()
        print('=' * 60)
        print(f'{site_name} — Fold 5 reference (2017–2024)')
        print('=' * 60)

        labels_f5 = res['fold5_ref']['labels']
        preds_f5  = res['fold5_ref']['preds']
        print('TRUTH')
        print(train_test_model.format_week_delimited(labels_f5, WEEKS_PER_YEAR))
        print('XGB PREDICTIONS')
        print(train_test_model.format_week_delimited(preds_f5, WEEKS_PER_YEAR))

        print()
        print(f'{site_name} — {EVAL_YEAR} evaluation')
        print('-' * 40)
        labels_2025 = res['labels_2025']
        preds_2025  = res['preds_raw_2025']
        print('TRUTH')
        print(train_test_model.format_week_delimited(labels_2025, WEEKS_PER_YEAR))
        print('XGB PREDICTIONS')
        print(train_test_model.format_week_delimited(preds_2025, WEEKS_PER_YEAR))

## Results

Prediction timeline and outcome counts for each site.  
Plots are saved to `../output/predictions_2025/<site>/plots/`.

In [ ]:
if not all_2025_results:
    print('Run the prediction cell first.')
else:
    for reef_location in SITES_TO_RUN:
        if reef_location not in all_2025_results:
            continue

        res       = all_2025_results[reef_location]
        site_name = res['site_name']
        labels    = res['labels_2025']
        masked    = res['masked_preds']
        m         = res['metrics_2025']
        n_weeks   = len(labels)
        weeks     = np.arange(SUMMER_START, SUMMER_START + n_weeks)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # ── Timeline: actual vs predicted ─────────────────────────────────────
        ax = axes[0]
        ax.plot(weeks, labels, 'bo-', label='Actual', markersize=6, linewidth=1.5)
        valid = ~np.isnan(masked)
        ax.plot(weeks[valid], masked[valid], 'rs--', label='XGB Predicted',
                markersize=6, linewidth=1.5, alpha=0.85)
        ax.axhline(y=VAL_NO_ONSET, color='gray', linestyle=':', alpha=0.6,
                   label=f'No onset ({VAL_NO_ONSET})')
        ax.set_xlabel('Summer Week', fontsize=12)
        ax.set_ylabel('Weeks Until Onset', fontsize=12)
        ax.set_title(f'{site_name} — {EVAL_YEAR} Predictions vs Actual', fontsize=13)
        ax.set_xticks(weeks)
        ax.legend(fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.4)

        # ── Outcome counts bar chart ──────────────────────────────────────────
        ax2 = axes[1]
        counts = [m['hit'], m['miss'], m['fa'], m['tno']]
        colors = ['#2ecc71', '#e67e22', '#e74c3c', '#3498db']
        bars = ax2.bar(['Hit', 'Miss', 'False Alarm', 'True\nNo-Onset'],
                       counts, color=colors, edgecolor='black', alpha=0.85)
        ax2.bar_label(bars, padding=2, fontsize=11)
        ax2.set_ylabel('Count', fontsize=12)
        ax2.set_title(f'{site_name} — {EVAL_YEAR} Outcome Counts', fontsize=13)

        f1_str = f"{m['f1']:.3f}" if np.isfinite(m['f1']) else 'N/A'
        oa_str = f"{m['onset_acc']:.0%}" if np.isfinite(m['onset_acc']) else 'N/A'
        fa_str = f"{m['fa_rate']:.0%}" if np.isfinite(m['fa_rate']) else 'N/A'
        ax2.set_xlabel(f'Onset Acc: {oa_str}  |  FA Rate: {fa_str}  |  F1: {f1_str}', fontsize=11)
        ax2.grid(axis='y', linestyle='--', alpha=0.4)

        plt.tight_layout()
        out_path = OUTPUT_DIR / OUT_EXP / reef_location / 'plots' / f'{reef_location}_{EVAL_YEAR}_predictions.png'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(str(out_path), dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved to {out_path}')

## Summary Table

In [ ]:
if not all_2025_results:
    print('Run the prediction cell first.')
else:
    rows = []
    for reef_location in SITES_TO_RUN:
        if reef_location not in all_2025_results:
            continue
        res = all_2025_results[reef_location]
        m   = res['metrics_2025']
        fv  = res['fold5_ref']

        f1_str = f"{m['f1']:.3f}" if np.isfinite(m['f1']) else 'N/A'
        oa_str = f"{m['onset_acc']:.0%}" if np.isfinite(m['onset_acc']) else 'N/A'
        fa_str = f"{m['fa_rate']:.0%}" if np.isfinite(m['fa_rate']) else 'N/A'

        rows.append({
            'Site':              res['site_name'],
            f'{EVAL_YEAR} Hit':  m['hit'],
            f'{EVAL_YEAR} Miss': m['miss'],
            f'{EVAL_YEAR} FA':   m['fa'],
            f'{EVAL_YEAR} TNO':  m['tno'],
            'Onset Acc':         oa_str,
            'FA Rate':           fa_str,
            f'{EVAL_YEAR} F1':   f1_str,
            'Fold5 F1 (2017-24)': f"{fv['f1']:.3f}",
        })

    df_summary = pd.DataFrame(rows).set_index('Site')
    display(df_summary)